# SDS 08 — Graph Partitioning, Modularity, and Girvan-Newman

Reusable reference notebook. Pure-Python cells explain the mechanics; Spark cells provide scalable preprocessing, modularity scoring, and a parallel reduced-core approximation using only built-in Spark/Python libraries.

## Recognition cues

- **partition / communities** → define an objective, not merely reachability
- **bridges** → edge betweenness
- **Girvan-Newman** → recompute betweenness after removals
- **justify K** → peak modularity
- **critical rule** → mutable `working_graph`, immutable `original_graph`

In [ ]:
from collections import defaultdict, deque, Counter
import copy, math, random

## Toy graph: two triangles joined by one bridge

In [ ]:
toy_edges=[('A','B'),('A','C'),('B','C'),('C','D'),('D','E'),('D','F'),('E','F')]

def adjacency_from_edges(edges):
    adj=defaultdict(set)
    for u,v in edges:
        if u==v: continue
        adj[u].add(v); adj[v].add(u)
    return {u:set(vs) for u,vs in adj.items()}

toy_adj=adjacency_from_edges(toy_edges)
toy_adj

## Connected components

In [ ]:
def connected_components(adj):
    seen=set(); comps=[]
    for s in adj:
        if s in seen: continue
        q=deque([s]); seen.add(s); comp=[]
        while q:
            u=q.popleft(); comp.append(u)
            for v in adj[u]:
                if v not in seen:
                    seen.add(v); q.append(v)
        comps.append(sorted(comp))
    return comps

connected_components(toy_adj)

## Modularity — ALWAYS score against the original graph

In [ ]:
def modularity(original_adj, communities):
    m=sum(len(vs) for vs in original_adj.values())/2.0
    if m==0: return 0.0
    deg={u:len(vs) for u,vs in original_adj.items()}
    Q=0.0
    for comm in communities:
        S=set(comm)
        # each internal edge is counted twice in this adjacency sum
        internal_twice=sum(1 for u in S for v in original_adj.get(u,set()) if v in S)
        Lc=internal_twice/2.0
        Dc=sum(deg.get(u,0) for u in S)
        Q += Lc/m - (Dc/(2*m))**2
    return Q

partition=[['A','B','C'],['D','E','F']]
print('Q two-community partition:', modularity(toy_adj, partition))
print('Expected:', 5/14)

## Brandes edge betweenness for an unweighted undirected graph

In [ ]:
def edge_betweenness_brandes(adj, sources=None):
    nodes=list(adj)
    if sources is None: sources=nodes
    eb=defaultdict(float)
    for s in sources:
        stack=[]
        pred={v:[] for v in nodes}
        sigma={v:0.0 for v in nodes}; sigma[s]=1.0
        dist={v:-1 for v in nodes}; dist[s]=0
        q=deque([s])
        while q:
            v=q.popleft(); stack.append(v)
            for w in adj[v]:
                if dist[w] < 0:
                    dist[w]=dist[v]+1; q.append(w)
                if dist[w] == dist[v]+1:
                    sigma[w]+=sigma[v]; pred[w].append(v)
        delta={v:0.0 for v in nodes}
        while stack:
            w=stack.pop()
            for v in pred[w]:
                if sigma[w] > 0:
                    c=(sigma[v]/sigma[w])*(1.0+delta[w])
                    e=tuple(sorted((v,w)))
                    eb[e]+=c
                    delta[v]+=c
    # if all sources are used, divide by 2 for undirected duplication
    if len(sources)==len(nodes):
        for e in list(eb): eb[e]/=2.0
    return dict(eb)

eb=edge_betweenness_brandes(toy_adj)
sorted(eb.items(), key=lambda kv:-kv[1])

The bridge `C-D` should have the largest edge betweenness.

## Exact Girvan-Newman on a small graph

In [ ]:
def remove_edge(adj,e):
    u,v=e
    adj[u].discard(v); adj[v].discard(u)

def girvan_newman_exact(original_adj, max_steps=None, remove_all_ties=False):
    original={u:set(vs) for u,vs in original_adj.items()}
    working={u:set(vs) for u,vs in original_adj.items()}
    best_part=connected_components(working)
    best_Q=modularity(original,best_part)
    history=[(0,len(best_part),best_Q,None)]
    step=0
    while True:
        remaining=sum(len(vs) for vs in working.values())//2
        if remaining==0: break
        if max_steps is not None and step>=max_steps: break
        eb=edge_betweenness_brandes(working)
        if not eb: break
        maxb=max(eb.values())
        tied=sorted([e for e,b in eb.items() if abs(b-maxb)<1e-12])
        to_remove=tied if remove_all_ties else [tied[0]]
        for e in to_remove: remove_edge(working,e)
        step += 1
        part=connected_components(working)
        Q=modularity(original,part)
        history.append((step,len(part),Q,to_remove))
        if Q>best_Q:
            best_Q=Q; best_part=copy.deepcopy(part)
    return best_part,best_Q,history

best,Q,hist=girvan_newman_exact(toy_adj)
print(best,Q)
for row in hist[:6]: print(row)

## Demonstration of the common bug: scoring against the damaged working graph

In [ ]:
working={u:set(vs) for u,vs in toy_adj.items()}
remove_edge(working,('C','D'))
part=connected_components(working)
print('Correct Q vs ORIGINAL:', modularity(toy_adj,part))
print('Wrong Q vs WORKING :', modularity(working,part))
print('The wrong score evaluates a different graph after the bridge has already been deleted.')

# Spark section

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark=SparkSession.builder.appName('SDS08-Girvan-Newman').getOrCreate()

## Load and canonicalize a simple undirected graph

In [ ]:
def load_clickstream_undirected(path='pageviews.csv'):
    raw=(spark.read.option('sep','\t').option('header',False).csv(path)
         .toDF('src','dst','type','count'))
    links=(raw.filter(F.col('type')=='link')
          .select('src','dst')
          .filter(F.col('src').isNotNull() & F.col('dst').isNotNull())
          .filter(F.col('src') != F.col('dst')))
    U=(links.select(F.least('src','dst').alias('u'),
                    F.greatest('src','dst').alias('v'))
       .distinct().cache())
    return U

## Distributed degree table and high-degree induced core

In [ ]:
def degree_table(U):
    return (U.select(F.col('u').alias('node'))
             .union(U.select(F.col('v').alias('node')))
             .groupBy('node').count()
             .withColumnRenamed('count','degree'))

def select_induced_core(U, K=300):
    deg=degree_table(U)
    top=deg.orderBy(F.desc('degree'),F.asc('node')).limit(K).select('node').cache()
    core=(U.join(top.select(F.col('node').alias('u')),'u')
           .join(top.select(F.col('node').alias('v')),'v')
           .select('u','v').distinct().cache())
    return top,core

## Convert ONLY a deliberately small reduced core to adjacency

In [ ]:
def collect_small_core_adjacency(core_edges):
    rows=core_edges.collect()   # use only after verifying the core is intentionally small
    return adjacency_from_edges([(r['u'],r['v']) for r in rows])

## Parallel sampled-source edge betweenness on a reduced core

In [ ]:
def _brandes_one_source(args):
    s, adj=args
    nodes=list(adj)
    stack=[]; pred={v:[] for v in nodes}
    sigma={v:0.0 for v in nodes}; sigma[s]=1.0
    dist={v:-1 for v in nodes}; dist[s]=0
    q=deque([s])
    while q:
        v=q.popleft(); stack.append(v)
        for w in adj[v]:
            if dist[w] < 0:
                dist[w]=dist[v]+1; q.append(w)
            if dist[w] == dist[v]+1:
                sigma[w]+=sigma[v]; pred[w].append(v)
    delta={v:0.0 for v in nodes}; out=[]
    while stack:
        w=stack.pop()
        for v in pred[w]:
            if sigma[w] > 0:
                c=(sigma[v]/sigma[w])*(1.0+delta[w])
                out.append((tuple(sorted((v,w))),c)); delta[v]+=c
    return out

def sampled_edge_betweenness_spark(sc, adjacency, sample_sources):
    # Safe only when adjacency is a deliberately small reduced graph.
    b=sc.broadcast(adjacency)
    rdd=sc.parallelize(list(sample_sources), max(1,min(len(sample_sources),sc.defaultParallelism)))
    totals=(rdd.flatMap(lambda s: _brandes_one_source((s,b.value)))
              .reduceByKey(lambda a,b:a+b)
              .collect())
    b.unpersist()
    return dict(totals)

## Reproducible approximate GN on a reduced core

In [ ]:
def girvan_newman_sampled_spark(sc, original_adj, sample_size=100, seed=42,
                                  max_steps=50, remove_top=1):
    original={u:set(vs) for u,vs in original_adj.items()}
    working={u:set(vs) for u,vs in original_adj.items()}
    rng=random.Random(seed)
    best=connected_components(working); best_Q=modularity(original,best)
    history=[]
    for step in range(max_steps):
        active=[u for u in working if working[u]]
        if len(active)<2: break
        sample=rng.sample(active,min(sample_size,len(active)))
        eb=sampled_edge_betweenness_spark(sc,working,sample)
        if not eb: break
        ranked=sorted(eb.items(),key=lambda kv:(-kv[1],kv[0]))
        # remove_top > 1 is an explicit approximation because scores become stale
        removed=[]
        for e,_ in ranked[:remove_top]:
            remove_edge(working,e); removed.append(e)
        part=connected_components(working)
        Q=modularity(original,part)
        history.append({'step':step+1,'K':len(part),'Q':Q,'removed':removed})
        if Q>best_Q:
            best_Q=Q; best=copy.deepcopy(part)
    return best,best_Q,history

## Distributed modularity scoring from assignments and ORIGINAL edges

In [ ]:
def modularity_spark(U, assignments):
    # U: canonical undirected original edge table (u,v)
    # assignments: (node, community)
    m=U.count()
    if m==0: return 0.0
    degree=degree_table(U)
    D=(degree.join(assignments,'node')
       .groupBy('community').agg(F.sum('degree').alias('D_c')))
    au=assignments.select(F.col('node').alias('u'),F.col('community').alias('cu'))
    av=assignments.select(F.col('node').alias('v'),F.col('community').alias('cv'))
    L=(U.join(au,'u').join(av,'v').filter(F.col('cu')==F.col('cv'))
       .groupBy(F.col('cu').alias('community')).count()
       .withColumnRenamed('count','L_c'))
    terms=(D.join(L,'community','left').fillna(0,subset=['L_c'])
           .withColumn('term', F.col('L_c')/F.lit(float(m)) -
                       (F.col('D_c')/F.lit(float(2*m)))**2))
    return float(terms.agg(F.sum('term').alias('Q')).first()['Q'] or 0.0)

## Convert a local partition to Spark assignments for distributed scoring

In [ ]:
def partition_to_assignments(partition):
    rows=[]
    for cid,comm in enumerate(partition):
        rows.extend((node,int(cid)) for node in comm)
    return spark.createDataFrame(rows,['node','community'])

## Exam checklist

1. State graph convention (undirected/simple/weighted?).
2. State approximation parameters: core K, betweenness sample size, random seed, edges removed per round.
3. Keep `original` and `working` separate.
4. Recompute betweenness after every exact removal; if removing top-k, call it approximate.
5. Choose K by peak modularity and report Q.
6. Keep big preprocessing/scoring in Spark.
7. Collect only a deliberately small core or final scalars.
8. Mention bias/resolution limitations.